In [ ]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)


In [ ]:
!apt-get install zip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zip is already the newest version (3.0-12build2).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import random
import shutil
from pathlib import Path

# Updated paths for your specific project
source = "/content/drive/MyDrive/PROOFAI/videodataset/extracted_frames"
target = "/content/drive/MyDrive/PROOFAI/videodataset/final_dataset"

# Your project classes
classes = ["fire", "smoke"]
split_ratio = 0.8

# Clean target directory to start fresh if needed
if os.path.exists(target):
    print(f"Cleaning existing target directory: {target}")
    for folder in ['train', 'val']:
        folder_path = os.path.join(target, folder)
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)

for cls in classes:
    # Locate images in the 'pos' subfolder created during extraction
    cls_path = Path(source) / cls / 'pos'
    if not cls_path.exists():
        print(f"Warning: Path {cls_path} not found. Skipping {cls}.")
        continue

    # Recursively find all image files
    images = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        images.extend(list(cls_path.rglob(ext)))

    # Sort to ensure reproducibility before shuffling
    images.sort()
    random.seed(42)
    random.shuffle(images)

    split_idx = int(len(images) * split_ratio)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    os.makedirs(os.path.join(target, 'train'), exist_ok=True)
    os.makedirs(os.path.join(target, 'val'), exist_ok=True)

    print(f"Copying {len(train_imgs)} training and {len(val_imgs)} validation images for {cls}...")

    # Copy files with unique names to prevent overwriting
    for img_list, folder in [(train_imgs, 'train'), (val_imgs, 'val')]:
        dest_dir = os.path.join(target, folder)
        for img_path in img_list:
            # Construct a unique name based on the video folder name and frame
            unique_name = f"{img_path.parent.parent.name}_{img_path.name}"
            shutil.copy2(str(img_path), os.path.join(dest_dir, unique_name))

print(f"Done! Your dataset is ready at: {target}")

Cleaning existing target directory: /content/drive/MyDrive/PROOFAI/videodataset/final_dataset
Copying 2533 training and 634 validation images for fire...
Copying 1859 training and 465 validation images for smoke...
Done! Your dataset is ready at: /content/drive/MyDrive/PROOFAI/videodataset/final_dataset


In [ ]:
import os
from pathlib import Path

base_path = "/content/drive/MyDrive/PROOFAI/videodataset/final_dataset"

def setup_yolo_folders(base_path):
    # YOLO expects structure:
    # dataset/images/train
    # dataset/labels/train
    for folder in ['train', 'val']:
        img_dir = Path(base_path) / 'images' / folder
        lbl_dir = Path(base_path) / 'labels' / folder
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        # Move images from the old flat structure to images/folder
        old_img_dir = Path(base_path) / folder
        if old_img_dir.exists() and old_img_dir.is_dir() and 'images' not in str(old_img_dir):
            print(f"Moving images from {old_img_dir} to {img_dir}...")
            for img in old_img_dir.glob('*'):
                if img.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    img.rename(img_dir / img.name)

def create_labels(base_path):
    for split in ['train', 'val']:
        img_dir = Path(base_path) / 'images' / split
        lbl_dir = Path(base_path) / 'labels' / split

        images = list(img_dir.glob('*'))
        print(f"Creating {len(images)} labels for {split}...")

        for img in images:
            # Determine class: 0 for fire, 1 for smoke based on filename
            class_id = 0 if 'fire' in img.name.lower() else 1
            # Create a box covering 80% of the image center as placeholder
            label_content = f"{class_id} 0.5 0.5 0.8 0.8"

            with open(lbl_dir / f"{img.stem}.txt", 'w') as f:
                f.write(label_content)

setup_yolo_folders(base_path)
create_labels(base_path)

# Update data.yaml with the NEW internal paths
yolo_yaml = f"""
train: {base_path}/images/train
val: {base_path}/images/val

nc: 2
names: ['fire', 'smoke']
"""

with open(os.path.join(base_path, 'data.yaml'), 'w') as f:
    f.write(yolo_yaml.strip())

print("Dataset structure corrected. Please run the training cell now.")

Moving images from /content/drive/MyDrive/PROOFAI/videodataset/final_dataset/train to /content/drive/MyDrive/PROOFAI/videodataset/final_dataset/images/train...
Moving images from /content/drive/MyDrive/PROOFAI/videodataset/final_dataset/val to /content/drive/MyDrive/PROOFAI/videodataset/final_dataset/images/val...
Creating 1153 labels for train...
Creating 647 labels for val...
Dataset structure corrected. Please run the training cell now.


In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os
import torch
from pathlib import Path

# 1. Configuration
YAML_PATH = "/content/drive/MyDrive/PROOFAI/videodataset/final_dataset/data.yaml"
DATASET_DIR = "/content/drive/MyDrive/PROOFAI/videodataset/final_dataset"

# 2. Force clear old cache files to ensure YOLO finds the new labels
print("Cleaning old cache files...")
for cache_file in Path(DATASET_DIR).rglob('*.cache'):
    try:
        os.remove(cache_file)
        print(f"Removed: {cache_file}")
    except Exception as e:
        print(f"Could not remove {cache_file}: {e}")

# 3. Verify label existence before starting
train_labels = list(Path(f"{DATASET_DIR}/labels/train").glob('*.txt'))
print(f"Verified: {len(train_labels)} label files found in train set.")

# 4. Initialize and Train
if len(train_labels) == 0:
    print("Error: No labels found. Please re-run the label generation cell (c9bb32b8).")
else:
    model = YOLO("yolo11n.pt")
    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"Starting training on {device}...")

    results = model.train(
        data=YAML_PATH,
        epochs=80,
        imgsz=640,
        batch=16,
        device=device,
        project='/content/drive/MyDrive/PROOFAI/training_results',
        name='fire_smoke_detection',
        exist_ok=True
    )

Cleaning old cache files...
Removed: /content/drive/MyDrive/PROOFAI/videodataset/final_dataset/train.cache
Verified: 1153 label files found in train set.
Starting training on 0...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/PROOFAI/videodataset/final_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_r

In [ ]:
!pip install -U ultralytics
from ultralytics import YOLO
import os

# Path to the best weights from your recent training
weights_path = '/content/drive/MyDrive/PROOFAI/training_results/fire_smoke_detection/weights/best.pt'

if os.path.exists(weights_path):
    # Load the trained model
    model = YOLO(weights_path)

    # Run validation on the test/val set
    print("Loading trained model for validation...")
    metrics = model.val()

    print("\nValidation Summary:")
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print("Validation completed successfully using saved weights.")
else:
    print(f"Error: Trained weights not found at {weights_path}. Please check your Drive path.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading trained model for validation...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 0.2±0.1 MB/s, size: 53.0 KB)
val: Scanning /content/drive/MyDrive/PROOFAI/videodataset/final_dataset/labels/val.cache... 647 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 647/647 113.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP

In [ ]:
# 1. Download a sample image for testing
!curl -o test_fire.jpg https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg
# Since the bus image doesn't have fire, let's use a placeholder that likely contains fire/smoke patterns or you can provide a URL.
# For demonstration, I will use an official fire sample URL if available, otherwise we test the logic.
!curl -o test_fire.jpg https://images.unsplash.com/photo-1534447677768-be436bb09401?q=80&w=1000&auto=format&fit=crop

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  134k  100  134k    0     0   586k      0 --:--:-- --:--:-- --:--:--  588k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2484k  100 2484k    0     0  11.1M      0 --:--:-- --:--:-- --:--:-- 11.1M
